# 1 – Imports 

I will import necessary libraries and define dataset location and image size.

In [1]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Folder with images
data_dir = "../images_data"

# Resize all images to 128x128 pixels
IMG_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

I import PyTorch libraries for deep learning and image handling.

transforms will help to preprocess images.

DEVICE allows using GPU if available, else CPU.

Constants define folder, image size, batch size, and number of epochs.

# 2 – Custom Dataset Class

I will load images and labels with PyTorch Dataset.

In [2]:
class ImageDataset(Dataset):
    def __init__(self, data_dir, folders, labels, transform=None):
        self.images = []
        self.labels = []
        self.transform = transform
        
        for folder, label in zip(folders, labels):
            path = os.path.join(data_dir, folder)
            for file in os.listdir(path):
                img_path = os.path.join(path, file)
                self.images.append(img_path)
                self.labels.append(label)
                
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label

I create a dataset class so PyTorch can handle images.

It reads all image paths and labels, converts images to RGB.

__getitem__ returns one image and its label for training.

# 3 – Define Transformations

I will resize and normalize images.

In [3]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),          # Converts to [0,1] tensor
])

Resize makes all images 128×128.

ToTensor converts images to PyTorch tensors for training.

#  4 – Create Dataset and DataLoader

In [4]:
folders = ["compliant", "non-compliant"]
labels = [1, 0]

dataset = ImageDataset(data_dir, folders, labels, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

DataLoader handles batches and shuffling for training.

# 5 – Define the CNN Model

In [ ]:
from torchvision import models

# Load pretrained ResNet (ImageNet weights)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace final layer
num_classes = 1  # binary classification
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(DEVICE)

# 6 – Loss and Optimizer

In [ ]:
# Loss and optimizer are created at the start of the training cell (next section).


BCEWithLogitsLoss matches binary labels to raw classifier outputs (logits); sigmoid is applied inside the loss.

Adam updates weights during training.

# Split the Data 

70% → Training

15% → Validation

15% → Test

In [8]:
from torch.utils.data import random_split

# Calculate sizes
total_size = len(dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

# Split dataset
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# 8 – Train the Model

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


history = {"loss": [], "accuracy": []}

for epoch in range(EPOCHS):
    total_loss = 0
    correct = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
    
    accuracy = correct / len(train_dataset)
    history["loss"].append(total_loss / len(train_loader))
    history["accuracy"].append(accuracy)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_loader):.4f} - Accuracy: {accuracy:.2f}")